In [1]:
import numpy as np
import pandas as pd




# Get data

In [2]:
hist = pd.read_csv("/Users/macbookpro/platform/Backend/data/processed/default_hist.csv")
orig = pd.read_csv('/Users/macbookpro/platform/Backend/data/raw/orig_data_col.csv')



/var/folders/gy/2mbpv0kx5jz22fry28cqlnkc0000gn/T/ipykernel_49966/2619987728.py:2: DtypeWarning: Columns (24,25,29) have mixed types. Specify dtype option on import or set low_memory=False.
  orig = pd.read_csv('/Users/macbookpro/platform/Backend/data/raw/orig_data_col.csv')


In [3]:
loan_hist = hist[hist['LOAN_SEQUENCE_NUMBER'] == 'F07Q10000071']
loan_orig = orig[orig['LOAN_SEQUENCE_NUMBER'] == 'F07Q10000071']


# Config

In [4]:
train_cfPath = '/Users/macbookpro/platform/Backend/configs/ressources/Lgd_class_train.yaml'
test_cfPath = '/Users/macbookpro/platform/Backend/configs/ressources/Lgd_class_test.yaml'

# Feature and scaler

In [5]:
import importlib
import src.LGDcomponent.pipelines.lgdFeaturePipeline as lgdFeaturePipeline
importlib.reload(lgdFeaturePipeline)
from src.LGDcomponent.pipelines.lgdFeaturePipeline import LGDFeaturePipeline

/Users/macbookpro/platform/Backend/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
pipeline = LGDFeaturePipeline(config_path=train_cfPath)

scaler config loaded successfully


In [7]:
X,y = pipeline.build(hist, orig)

Copy DataFrame     : 1.5s
Cast DPD           : 0.1s
Groupby            : 0.0s
Colonnes de travail: 0.9s
[LGD] Loans exclus pour EAD=0 (artefact de séquence) : 196
[LGD] Loans avec target calculée : 30807 / 30807
[LGD] Observations clippées hors [0,1] : 4238
[LGD] Distribution LGD :
count    30807.0000
mean         0.3668
std          0.2918
min          0.0000
25%          0.1260
50%          0.3288
75%          0.5417
max          1.0000
Name: lgd_target, dtype: float64
🏃 View run LGD_scaler_preprocessing_v1 at: http://localhost:5000/#/experiments/3/runs/0d46ba18eb30471d8a0809e49673ad98
🧪 View experiment at: http://localhost:5000/#/experiments/3
datas scaled


# Split and save

In [8]:
from sklearn.model_selection import train_test_split

# 1. Separate the full dataset into Training (80%) and a temporary Test set (20%)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# 2. Split the temp set into Training (70%) and Validation (30% of the temp set, or 15% of the total)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.30, random_state=42)

In [ ]:
#X_train.to_csv('/Users/macbookpro/platform/Backend/data/processed/LGD/x_train.csv',sep=',',index=False)
#X_val.to_csv('/Users/macbookpro/platform/Backend/data/processed/LGD/x_val.csv',sep=',',index=False)
#X_test.to_csv('/Users/macbookpro/platform/Backend/data/processed/LGD/x_test.csv',sep=',',index=False)

In [ ]:
#pd.DataFrame(y_train).to_csv('/Users/macbookpro/platform/Backend/data/processed/LGD/y_train.csv',sep=',',index=False)
#pd.DataFrame(y_val).to_csv('/Users/macbookpro/platform/Backend/data/processed/LGD/y_val.csv',sep=',',index=False)
#pd.DataFrame(y_test).to_csv('/Users/macbookpro/platform/Backend/data/processed/LGD/y_test.csv',sep=',',index=False)#

# Run

train

In [9]:

import src.LGDcomponent.run.lgbm_multiclass as lgbm
importlib.reload(lgbm)

from src.LGDcomponent.run.lgbm_multiclass import LgbmMulticlass as LGBM_Multiclass

In [10]:
train_map = {'x_train':X_train, 'y_train':y_train}
val_map = {'x_val':X_val, 'y_val':y_val}

In [11]:
train = LGBM_Multiclass(train_map = train_map, val_map = val_map, config_path = train_cfPath,test_path = test_cfPath)

In [12]:
train.run()

[I 2026-07-01 22:51:56,758] A new study created in memory with name: no-name-eaf71f7d-9b96-409e-883e-ce581301cd87


[LGDDiscretizer] n_bins ajusté : 8 demandés →  7 bins effectifs (doublons dans la distribution).


[I 2026-07-01 22:51:58,196] Trial 0 finished with value: -0.2684438328186238 and parameters: {'max_depth': 5, 'num_leaves': 67, 'min_child_samples': 27, 'reg_lambda': 9.00157681631971, 'reg_alpha': 0.48562220486406016, 'subsample': 0.9114016033020838, 'colsample_bytree': 0.9698458465790186, 'learning_rate': 0.04692639864829361, 'n_estimators': 145}. Best is trial 0 with value: -0.2684438328186238.


RMSE: 0.2684 | Dxy: 0.2990 | ECE: 0.0166


[I 2026-07-01 22:52:00,154] Trial 1 finished with value: -0.26861983737663986 and parameters: {'max_depth': 5, 'num_leaves': 26, 'min_child_samples': 34, 'reg_lambda': 9.208960899044447, 'reg_alpha': 0.5793102322815727, 'subsample': 0.8676426293477928, 'colsample_bytree': 0.6395232242577493, 'learning_rate': 0.04929778503848507, 'n_estimators': 125}. Best is trial 1 with value: -0.26861983737663986.


...

RMSE: 0.2715 | Dxy: 0.2833 | ECE: 0.0194


[I 2026-07-01 22:52:34,799] Trial 18 finished with value: -0.278672500253441 and parameters: {'max_depth': 4, 'num_leaves': 29, 'min_child_samples': 50, 'reg_lambda': 9.9469059053736, 'reg_alpha': 0.6538384098941046, 'subsample': 0.8496194785498699, 'colsample_bytree': 0.9478759899513705, 'learning_rate': 0.0100785037972742, 'n_estimators': 109}. Best is trial 18 with value: -0.278672500253441.


RMSE: 0.2787 | Dxy: 0.2614 | ECE: 0.0316


[I 2026-07-01 22:52:35,797] Trial 19 finished with value: -0.2780277225239494 and parameters: {'max_depth': 4, 'num_leaves': 28, 'min_child_samples': 20, 'reg_lambda': 9.95447136759794, 'reg_alpha': 0.1431394255514266, 'subsample': 0.8275997164144597, 'colsample_bytree': 0.9529545813882018, 'learning_rate': 0.01196185134783885, 'n_estimators': 101}. Best is trial 18 with value: -0.278672500253441.


RMSE: 0.2780 | Dxy: 0.2629 | ECE: 0.0320
🏃 View run LightGBM LGD Multiclass Train at: http://localhost:5000/#/experiments/3/runs/d8fa9bbc74de4aab83c8437d7b254dd5
🧪 View experiment at: http://localhost:5000/#/experiments/3


test

In [13]:
bin_edges = train.discretizer.bin_edges_

In [14]:
bin_edges

array([0.        , 0.12463379, 0.23087378, 0.32667991, 0.42833108,
       0.53819802, 0.72005435, 1.        ])

In [15]:
test_map ={'x_test':X_test, 'y_test':y_test}

In [16]:
test = LGBM_Multiclass(test_map=test_map, config_path = test_cfPath)

In [17]:
test.run()

🏃 View run LightGBM LGD Multiclass Test at: http://localhost:5000/#/experiments/3/runs/8316d31a0dc045fb8883807f2969b2b2
🧪 View experiment at: http://localhost:5000/#/experiments/3


In [18]:
test_discretizer = test.discretizer

In [19]:
test_discretizer.bin_edges_

array([0.        , 0.12463379, 0.23087378, 0.32667991, 0.42833108,
       0.53819802, 0.72005435, 1.        ])

# compute lgd

In [ ]:
#midpoints = (bin_edges[:-1] + bin_edges[1:]) / 2
# [0.0623, 0.1778, 0.2788, 0.3775, 0.4832, 0.6291, 0.8600]

#y_predict = sum(proba[0] * midpoints)
# = 0.1673*0.0623 + 0.0426*0.1778 + 0.0483*0.2788 + 0.0613*0.3775
#   + 0.0533*0.4832 + 0.0908*0.6291 + 0.5363*0.8600
# ≈ 0.587

# Test inference

In [20]:
import src.LGDcomponent.LgdPrediction as lgd
importlib.reload(lgd)
from src.LGDcomponent.LgdPrediction import LGDPrediction

In [21]:
mlflow_config ='/Users/macbookpro/platform/Backend/configs/Lgd_mlFlow_config.yaml'
model_config = '/Users/macbookpro/platform/Backend/configs/Lgd_model_config.yaml'

In [22]:
inference = LGDPrediction(hist=loan_hist,orig=loan_orig,mlflow_config=mlflow_config, model_config=model_config)

Copy DataFrame     : 0.0s
Cast DPD           : 0.0s
Groupby            : 0.0s
Colonnes de travail: 0.0s


In [23]:
discretize = inference.discretizer

In [24]:
type(discretize)

pipelines.Features.Lgd_discretizer.LGDDiscretizer

In [25]:
inference.apply()

[np.float64(0.4470168693308959)]